# AIDA 1141 Final Project – 02 Baselines and Model

**Data version 2** (`shared/synthetic-data/`), using the split and inputs fixed in notebook 01. This notebook compares a "never at risk" reference, a days-until-start rule, and a logistic regression on the **validation** set. Every threshold is chosen on training data only, and the test set is not used.

All records are synthetic; results describe methods applied to simulated outcomes, not real employees.

In [1]:
import hashlib
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_VERSION = "2"
SD = Path("../../../shared/synthetic-data")          # data version 2
OUTPUT_DIR = Path("../outputs")
SEED = 2026
RECALL_TARGET = 0.90

read = lambda path: pd.read_csv(path, dtype=str, keep_default_na=False)
requests = read(SD / "cleaning/output/requests_cleaned.csv")
assessments = read(SD / "requests/data/assessments.csv")
employee_split = read(OUTPUT_DIR / "employee_split.csv")          # from notebook 01 (protected)
feature_list = read(OUTPUT_DIR / "feature_list.csv")              # from notebook 01
split_summary_01 = pd.read_csv(OUTPUT_DIR / "split_summary.csv", index_col="split")

# Fingerprints of the exact inputs, saved with the model settings so later notebooks can identify them
SOURCES = {"requests_cleaned.csv": SD / "cleaning/output/requests_cleaned.csv",
           "assessments.csv": SD / "requests/data/assessments.csv",
           "employee_split.csv": OUTPUT_DIR / "employee_split.csv", "feature_list.csv": OUTPUT_DIR / "feature_list.csv"}
fingerprints = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in SOURCES.items()}

FEATURES = feature_list["feature"].tolist()
NUMERIC = feature_list.loc[feature_list["type"] == "numeric", "feature"].tolist()
CATEGORICAL = feature_list.loc[feature_list["type"] == "categorical", "feature"].tolist()

# Rebuild the eligible early-warning assessments exactly as defined in notebook 01
data = (assessments
        .merge(requests[["request_id", "employee_id", "task_type", "hiring_department", "employee_type", "intake_channel",
                         "info_issue_at_intake"]].rename(columns={"employee_id": "request_employee_id"}),
               on="request_id", how="left", validate="many_to_one")
        .merge(employee_split[["employee_id", "split"]], on="employee_id", how="left", validate="many_to_one"))
data["intake_channel"] = data["intake_channel"].replace("", "unknown")   # missing at source (notebook 01)
data[NUMERIC] = data[NUMERIC].astype(int)
early = data[(data["at_risk"] != "") & (data["past_deadline_at_assessment"] == "False")].copy()
early["y"] = (early["at_risk"] == "True").astype(int)
overdue = data[(data["at_risk"] != "") & (data["past_deadline_at_assessment"] == "True")].copy()
print(f"Data version {DATA_VERSION}; scikit-learn {sklearn.__version__}; "
      f"{len(early):,} eligible early-warning assessments, {len(overdue):,} overdue")

Data version 2; scikit-learn 1.9.1; 5,173 eligible early-warning assessments, 191 overdue


## A. Consistency with notebook 01

The eligible assessments are rebuilt from the same files and rules as notebook 01. Matching totals alone would not prove the same records were loaded, so the checks also confirm complete, unique assessment IDs, employee IDs that match their requests, documented label and overdue values, one label per task, and no overlap between training and validation employees. The early-warning, at-risk, and overdue counts must then match notebook 01's saved split summary in every split, no model input may be missing or empty, and `unknown` must be the only value added to the intake channel.

Fingerprints (SHA-256) of the cleaned requests, assessments, split, and feature list are saved with the model settings, so later notebooks can confirm they use exactly the same inputs. The checks are saved to `outputs/checks_02_consistency.csv`.

In [2]:
by_split = early.groupby("split")
train_ids = set(early.loc[early["split"] == "train", "employee_id"])
validation_ids = set(early.loc[early["split"] == "validation", "employee_id"])
checks = pd.Series({
    "Assessment IDs are complete and unique": data["assessment_id"].ne("").all() and data["assessment_id"].is_unique,
    "Assessment employee IDs match their requests": (data["employee_id"] == data["request_employee_id"]).all(),
    "at_risk is True, False, or blank only": data["at_risk"].isin(["True", "False", ""]).all(),
    "past_deadline_at_assessment is True or False only": data["past_deadline_at_assessment"].isin(["True", "False"]).all(),
    "Each task has one label across its checkpoints": data.groupby("request_id")["at_risk"].nunique().max() == 1,
    "Every assessment has a split from notebook 01": data["split"].notna().all(),
    "Training and validation employees do not overlap": not train_ids & validation_ids,
    "Eligible early-warning counts match notebook 01, by split":
        (by_split.size().reindex(split_summary_01.index) == split_summary_01["early_warning_assessments"]).all(),
    "At-risk counts match notebook 01, by split":
        (by_split["y"].sum().reindex(split_summary_01.index) == split_summary_01["early_warning_at_risk"]).all(),
    "Overdue counts match notebook 01, by split":
        (overdue.groupby("split").size().reindex(split_summary_01.index, fill_value=0)
         == split_summary_01["overdue_assessments"]).all(),
    "No missing or empty model inputs": early[FEATURES].notna().all().all() and early[FEATURES].ne("").all().all(),
    "'unknown' is the only added intake-channel value":
        set(early["intake_channel"]) <= {"web_form", "email", "phone", "unknown"},
}, name="passed")
assert checks.all(), "The rebuilt data does not match notebook 01."
checks.rename("passed").to_csv(OUTPUT_DIR / "checks_02_consistency.csv", header=True)
train, validation = early[early["split"] == "train"], early[early["split"] == "validation"]
print(f"Training: {len(train):,} assessments ({train['y'].sum()} at risk); "
      f"validation: {len(validation):,} ({validation['y'].sum()} at risk). Test data is not used in this notebook.")
checks.to_frame()

Training: 3,084 assessments (563 at risk); validation: 1,033 (194 at risk). Test data is not used in this notebook.


,passed
Assessment IDs are complete and unique,True
Assessment employee IDs match their requests,True
"at_risk is True, False, or blank only",True
past_deadline_at_assessment is True or False only,True
Each task has one label across its checkpoints,True
Every assessment has a split from notebook 01,True
Training and validation employees do not overlap,True
"Eligible early-warning counts match notebook 01, by split",True
"At-risk counts match notebook 01, by split",True
"Overdue counts match notebook 01, by split",True


## B. Days-until-start rule

The rule flags an assessment when the number of business days until the start date is at or below a threshold. The threshold is the smallest count that reaches 90% recall on the **training** early-warning assessments, so validation data plays no part in choosing it. A "never at risk" reference is also reported, to show what accuracy alone would hide.

In [3]:
def smallest_days_threshold(rows):
    """Smallest k where flagging 'days until start <= k' reaches the recall target on these rows."""
    at_risk_days = rows.loc[rows["y"] == 1, "business_days_until_start"]
    for k in range(rows["business_days_until_start"].min(), rows["business_days_until_start"].max() + 1):
        if (at_risk_days <= k).mean() >= RECALL_TARGET:
            return int(k)
    raise ValueError("No threshold reaches the recall target.")


DAYS_THRESHOLD = smallest_days_threshold(train)                     # chosen on training data only
training_recall = (train.loc[train["y"] == 1, "business_days_until_start"] <= DAYS_THRESHOLD).mean()
print(f"Days-until-start rule: flag when days until start <= {DAYS_THRESHOLD} "
      f"(training recall {training_recall:.1%})")

Days-until-start rule: flag when days until start <= 8 (training recall 91.7%)


## C. Logistic regression: threshold from grouped cross-validation

The model is a logistic regression in one pipeline with scaling of numeric inputs and one-hot encoding of categories, so every fit uses only the rows it is given. To choose a flag threshold without touching validation data, five-fold cross-validation is run **within training**, grouped by employee so no employee appears in both the fitting and held-out rows of a fold. Before the threshold is chosen, the checks confirm that every training assessment received exactly one out-of-fold prediction and that every probability is finite and between 0 and 1. The threshold is the highest out-of-fold probability whose flags reach 90% recall.

**Unseen categories.** The encoder drops each feature's first category as the reference and encodes any category not seen during fitting as all zeros. As a result, an unseen category receives the same encoding as the reference category, so the model would treat it like the reference. The fold table records whether each fitting fold contains the `unknown` intake channel and lists any held-out categories missing from its fitting rows; the split is not changed to force any category into training.

The out-of-fold predictions, fold table, and cross-validation checks are saved.

In [4]:
def make_pipeline():
    """Scaling, encoding, and logistic regression in one pipeline, so every fit uses only the rows it is given."""
    prepare = ColumnTransformer([
        ("numeric", StandardScaler(), NUMERIC),
        ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore"), CATEGORICAL),
    ])
    return Pipeline([("prepare", prepare), ("model", LogisticRegression(max_iter=1000))])


X_train, y_train = train[FEATURES], train["y"]
folds = GroupKFold(n_splits=5, shuffle=True, random_state=SEED)
out_of_fold = pd.Series(np.nan, index=train.index)
prediction_count = np.zeros(len(train), dtype=int)
fold_rows = []
for fold, (fit_idx, held_idx) in enumerate(folds.split(X_train, y_train, groups=train["employee_id"]), start=1):
    fit_rows, held_rows = train.iloc[fit_idx], train.iloc[held_idx]
    assert fit_rows["y"].nunique() == 2, "A fitting fold does not contain both outcome classes."
    fitted = make_pipeline().fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])
    out_of_fold.iloc[held_idx] = fitted.predict_proba(X_train.iloc[held_idx])[:, 1]
    prediction_count[held_idx] += 1
    # Categories in the held-out rows that the fitting rows never saw are encoded as all zeros (handle_unknown)
    unseen = {c: sorted(set(held_rows[c]) - set(fit_rows[c])) for c in CATEGORICAL}
    fold_rows.append({"fold": fold, "fit_assessments": len(fit_idx), "fit_at_risk": int(fit_rows["y"].sum()),
                      "held_out_assessments": len(held_idx), "held_out_at_risk": int(held_rows["y"].sum()),
                      "employees_in_both": len(set(fit_rows["employee_id"]) & set(held_rows["employee_id"])),
                      "fit_has_unknown_channel": "unknown" in set(fit_rows["intake_channel"]),
                      "held_out_categories_unseen_in_fit": "; ".join(f"{c}: {v}" for c, v in unseen.items() if v) or "none"})
fold_summary = pd.DataFrame(fold_rows)

# Check the out-of-fold predictions before using them to choose the threshold
prediction_checks = pd.Series({
    "Every training assessment has exactly one out-of-fold prediction": (prediction_count == 1).all(),
    "Every out-of-fold probability is finite and between 0 and 1":
        np.isfinite(out_of_fold).all() and out_of_fold.between(0, 1).all(),
    "No employee appears in both the fitting and held-out rows of a fold": (fold_summary["employees_in_both"] == 0).all(),
}, name="passed")
assert prediction_checks.all(), "A cross-validation check failed."

# Threshold: the highest out-of-fold probability whose ">= threshold" flags reach 90% training recall
scores = np.sort(out_of_fold.unique())
MODEL_THRESHOLD = float(max(t for t in scores if (out_of_fold[y_train == 1] >= t).mean() >= RECALL_TARGET))
cv_checks = pd.concat([prediction_checks, pd.Series({
    "Out-of-fold recall at the threshold reaches the target": (out_of_fold[y_train == 1] >= MODEL_THRESHOLD).mean() >= RECALL_TARGET,
}, name="passed")])
assert cv_checks.all(), "A cross-validation check failed."
cv_checks.rename("passed").to_csv(OUTPUT_DIR / "checks_02_cross_validation.csv", header=True)
train.assign(out_of_fold_probability=out_of_fold)[["assessment_id", "request_id", "employee_id", "checkpoint", "y",
                                                   "out_of_fold_probability"]].rename(columns={"y": "at_risk"}).to_csv(
    OUTPUT_DIR / "train_out_of_fold_predictions.csv", index=False)
fold_summary.to_csv(OUTPUT_DIR / "cv_fold_summary.csv", index=False)
print(f"Model threshold: {MODEL_THRESHOLD:.4f} (full precision saved in model_settings.json); "
      f"out-of-fold training recall {(out_of_fold[y_train == 1] >= MODEL_THRESHOLD).mean():.1%}")
display(cv_checks.to_frame())
fold_summary

Model threshold: 0.1338 (full precision saved in model_settings.json); out-of-fold training recall 90.1%


,passed
Every training assessment has exactly one out-of-fold prediction,True
Every out-of-fold probability is finite and between 0 and 1,True
No employee appears in both the fitting and held-out rows of a fold,True
Out-of-fold recall at the threshold reaches the target,True


,fold,fit_assessments,fit_at_risk,held_out_assessments,held_out_at_risk,employees_in_both,fit_has_unknown_channel,held_out_categories_unseen_in_fit
0,1,2467,447,617,116,0,True,none
1,2,2448,433,636,130,0,True,none
2,3,2446,457,638,106,0,True,none
3,4,2494,453,590,110,0,True,none
4,5,2481,462,603,101,0,True,none


## D. Final model and settings

The same pipeline is refitted once on all training early-warning assessments. The notebook saves the fitted model; the full-precision thresholds and how they were chosen; the estimator's actual parameters and the encoder's handling of dropped and unseen categories; whether `unknown` appears in training; the input fingerprints; and package versions. It also saves the training mean, scaling divisor, and standard deviation for each numeric input, the categories seen in training with each one's reference category, and the coefficients. Coefficients describe associations in the simulated data, not causes.

In [5]:
model = make_pipeline().fit(X_train, y_train)           # final model: all training early-warning assessments

prepare = model.named_steps["prepare"]
scaler = prepare.named_transformers_["numeric"]
encoder = prepare.named_transformers_["categorical"]
numeric_scaling = pd.DataFrame({"feature": NUMERIC, "training_mean": scaler.mean_,
                                "training_scale": scaler.scale_,          # divisor used for scaling
                                "training_sd": np.sqrt(scaler.var_)})     # actual standard deviation
categories = pd.DataFrame([{"feature": f, "reference_category": str(cats[0]),
                            "categories_seen_in_training": "; ".join(map(str, cats))}
                           for f, cats in zip(CATEGORICAL, encoder.categories_)])
unknown_channel_in_training = "unknown" in encoder.categories_[CATEGORICAL.index("intake_channel")]   # recorded, not forced

names = prepare.get_feature_names_out()
coefficients = pd.DataFrame({"term": names, "coefficient": model.named_steps["model"].coef_[0]})
coefficients["feature"] = [next(f for f in FEATURES if n.split("__", 1)[1].startswith(f)) for n in names]
reference = dict(zip(categories["feature"], categories["reference_category"]))
coefficients["compared_with"] = ["1 training standard deviation increase" if f in NUMERIC
                                 else f"reference: {reference[f]}" for f in coefficients["feature"]]
coefficients = pd.concat([pd.DataFrame([{"term": "intercept", "coefficient": model.named_steps["model"].intercept_[0],
                                         "feature": "", "compared_with": ""}]), coefficients], ignore_index=True)

settings = {
    "data_version": DATA_VERSION, "model": "logistic regression (L2, C=1, unweighted), scaled numeric inputs, "
    "one-hot categories (first category dropped; unseen categories encoded as zeros)",
    "features": FEATURES, "recall_target": RECALL_TARGET, "model_threshold": MODEL_THRESHOLD,
    "model_threshold_rule": "highest out-of-fold probability (5-fold, grouped by employee, training only) reaching the recall target",
    "days_threshold": DAYS_THRESHOLD, "days_threshold_rule": "smallest days-until-start count reaching the recall target on training",
    "unknown_channel_in_training": bool(unknown_channel_in_training),
    "estimator_parameters": model.named_steps["model"].get_params(),
    "encoder_parameters": {k: encoder.get_params()[k] for k in ["drop", "handle_unknown"]},
    "source_fingerprints_sha256": fingerprints,
    "seed": SEED, "package_versions": {"scikit-learn": sklearn.__version__, "pandas": pd.__version__,
                                       "numpy": np.__version__, "scipy": scipy.__version__, "joblib": joblib.__version__},
}
joblib.dump(model, OUTPUT_DIR / "model.joblib")
(OUTPUT_DIR / "model_settings.json").write_text(json.dumps(settings, indent=2, default=str))
numeric_scaling.to_csv(OUTPUT_DIR / "numeric_scaling.csv", index=False)
categories.to_csv(OUTPUT_DIR / "category_encoding.csv", index=False)
coefficients.round(4).to_csv(OUTPUT_DIR / "coefficients.csv", index=False)
display(categories)
coefficients.round(3)

,feature,reference_category,categories_seen_in_training
0,task_type,equipment_setup,equipment_setup; orientation_scheduling; payro...
1,hiring_department,Administration,Administration; Finance; Parks; Recreation; Tr...
2,employee_type,permanent,permanent; seasonal; temporary
3,intake_channel,email,email; phone; unknown; web_form
4,info_issue_at_intake,False,False; True
5,current_status,in_progress,in_progress; open


,term,coefficient,feature,compared_with
0,intercept,-5.107,,
1,numeric__business_days_until_start,-4.544,business_days_until_start,1 training standard deviation increase
2,numeric__business_days_in_current_status,0.194,business_days_in_current_status,1 training standard deviation increase
3,numeric__team_open_workload,0.203,team_open_workload,1 training standard deviation increase
4,categorical__task_type_orientation_scheduling,-0.203,task_type,reference: equipment_setup
5,categorical__task_type_payroll_setup,0.037,task_type,reference: equipment_setup
6,categorical__task_type_system_access,-0.656,task_type,reference: equipment_setup
7,categorical__task_type_workplace_access,0.035,task_type,reference: equipment_setup
8,categorical__hiring_department_Finance,0.438,hiring_department,reference: Administration
9,categorical__hiring_department_Parks,0.116,hiring_department,reference: Administration


## E. Validation comparison

Each method is applied to the validation early-warning assessments. The table reports confusion counts, recall (the primary measure), precision, flagged assessments, and two task-level measures: unique tasks flagged (review volume) and at-risk tasks that never received a warning, which assessment recall alone does not show. Ranking measures (ROC AUC and average precision) describe how well each score orders assessments by risk, independent of the threshold; the notebook confirms validation contains both outcome classes before calculating them. Overdue assessments are already late and are reported separately, and saved to `outputs/validation_overdue.csv`.

Validation predictions are saved for the threshold and review-queue analysis (notebook 03) and the error and bias review (notebook 04).

In [6]:
validation = validation.assign(
    risk_probability=model.predict_proba(validation[FEATURES])[:, 1],
    never_at_risk=False,
    days_rule=validation["business_days_until_start"] <= DAYS_THRESHOLD)
validation["logistic_regression"] = validation["risk_probability"] >= MODEL_THRESHOLD
METHODS = {"never_at_risk": "Never at risk (reference)", "days_rule": f"Days until start <= {DAYS_THRESHOLD}",
           "logistic_regression": "Logistic regression"}


def evaluate(rows, flag):
    """Assessment-level counts and measures, plus task-level review volume and missed at-risk tasks."""
    y, p = rows["y"] == 1, rows[flag]
    tp, fp, fn, tn = int((p & y).sum()), int((p & ~y).sum()), int((~p & y).sum()), int((~p & ~y).sum())
    tasks = rows.assign(flag=p).groupby("request_id").agg(at_risk=("y", "max"), flagged=("flag", "any"))
    return {"assessments": len(rows), "at_risk": int(y.sum()), "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "recall": tp / (tp + fn) if tp + fn else np.nan, "precision": tp / (tp + fp) if tp + fp else np.nan,
            "flagged_assessments": int(p.sum()), "flagged_%": p.mean() * 100,
            "unique_tasks_flagged": int(tasks["flagged"].sum()), "at_risk_tasks": int(tasks["at_risk"].sum()),
            "at_risk_tasks_never_flagged": int((tasks["at_risk"].astype(bool) & ~tasks["flagged"]).sum())}


assert len(validation) and validation["y"].nunique() == 2, "Validation needs both classes for these measures."
comparison = pd.DataFrame([evaluate(validation, m) for m in METHODS], index=list(METHODS.values()))
comparison.index.name = "method"
ranking = pd.DataFrame({
    "ROC AUC": [roc_auc_score(validation["y"], validation["risk_probability"]),
                roc_auc_score(validation["y"], -validation["business_days_until_start"])],
    "average precision": [average_precision_score(validation["y"], validation["risk_probability"]),
                          average_precision_score(validation["y"], -validation["business_days_until_start"])],
}, index=["Logistic regression (probability)", "Days until start (fewer days = higher risk)"])

# Overdue validation assessments are already late, so they are reported separately
overdue_val = overdue[overdue["split"] == "validation"]
overdue_flags = pd.Series({
    "assessments": len(overdue_val),
    "flagged by the days rule": int((overdue_val["business_days_until_start"] <= DAYS_THRESHOLD).sum()),
    "flagged by the model": (int((model.predict_proba(overdue_val[FEATURES])[:, 1] >= MODEL_THRESHOLD).sum())
                             if len(overdue_val) else 0),       # an empty table cannot be scored
}, name="overdue validation assessments")
overdue_flags.to_frame().to_csv(OUTPUT_DIR / "validation_overdue.csv")

comparison.insert(0, "data_version", DATA_VERSION)
comparison.to_csv(OUTPUT_DIR / "validation_comparison.csv")
ranking.round(4).to_csv(OUTPUT_DIR / "validation_ranking.csv")
validation[["assessment_id", "request_id", "employee_id", "checkpoint", "task_type", "employee_type", "y",
            "business_days_until_start", "risk_probability", "days_rule", "logistic_regression"]].rename(
    columns={"y": "at_risk"}).to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)
display(comparison.round(3))
display(ranking.round(3))
overdue_flags.to_frame()

,data_version,assessments,at_risk,TP,FP,FN,TN,recall,precision,flagged_assessments,flagged_%,unique_tasks_flagged,at_risk_tasks,at_risk_tasks_never_flagged
method,,,,,,,,,,,,,,
Never at risk (reference),2,1033,194,0,0,194,839,0.000,NaN,0,0.000,0,154,154
Days until start <= 8,2,1033,194,177,309,17,530,0.912,0.364,486,47.047,366,154,0
Logistic regression,2,1033,194,178,316,16,523,0.918,0.360,494,47.822,368,154,0


,ROC AUC,average precision
Logistic regression (probability),0.885,0.707
Days until start (fewer days = higher risk),0.876,0.643


,overdue validation assessments
assessments,24
flagged by the days rule,24
flagged by the model,24


## F. What the validation results show

**Thresholds.** Both thresholds were chosen on training data: the days rule flags assessments with 8 or fewer business days until the start date (91.7% training recall), and the model flags estimated risks of about 0.134 or higher (90.1% out-of-fold training recall).

**Validation.** Of 1,033 validation early-warning assessments, 194 were at risk. The days rule reached 91.2% recall (177 of 194) and 36.4% precision, flagging 486 assessments (47.0%) across 366 tasks. The model reached 91.8% recall (178 of 194) and 36.0% precision, flagging 494 assessments (47.8%) across 368 tasks. Neither method left any of the 154 at-risk validation tasks without a warning, and both flagged all 24 overdue assessments.

**Interpretation.** This is a mixed result. At the selected thresholds, the model had slightly higher recall but lower precision and more flags, so no reduction in review volume was established. Its ranking scores were higher (ROC AUC 0.885 vs 0.876; average precision 0.707 vs 0.643), which may support a useful review queue, but ranking measures alone do not show that; notebook 03 will examine how many at-risk cases appear near the top of the queue and the total review volume. The simple days rule remains a competitive baseline.

**Limits.** These are validation results used for development, not final test evidence. They describe simulated outcomes in data version 2, which differs from version 1 in its process assumptions, sample, period, and split as well as in recording problems, so the change from version 1 results cannot be attributed to any single difference.